## Part 3: Features engineering, split and preprocessing (P3)

### Section 3.1: Setup

In [ ]:
# LOCAL SETUP (removed automatically when the notebooks are merged)
import sys
sys.path.append("../src")
from f1_clean import load_clean, SEED

In [ ]:
clean = load_clean()
base     = clean["model_base"]   # pre-race columns + target 'podium'
outcomes = clean["outcomes"]     # post-race columns NEVER use as features
tables   = clean["tables"]       # all other cleaned tables

base.shape, outcomes.shape, list(tables.keys())

### Section 3.2: Base table sanity check

The dataset has been cleaned with 0 duplicated rows for (raceId, driverId). The missing values are present in the qualifying-related columns q1_s, q2_s, q3_s, qual_position.

In [ ]:
import pandas as pd

# Missingness across the full base table (all columns, all years)
miss = base.isna().mean().sort_values(ascending=False)

summary = pd.DataFrame({
    "missing_count": base.isna().sum(),
    "missing_share": miss,
    "dtype": base.dtypes,
})
summary["missing_share"] = summary["missing_share"].round(4)

print("=" * 65)
print(f"base shape: {base.shape[0]:,} rows × {base.shape[1]} columns")
print(f"Columns with ANY missing:  {(summary['missing_count'] > 0).sum()}")
print(f"Columns with NO  missing:  {(summary['missing_count'] == 0).sum()}")
print("=" * 65)

print("\nColumns with missing values (sorted):")
print(summary[summary["missing_count"] > 0].to_string())

In [ ]:
# Grain and target check: one row per (raceId, driverId), podium is 0/1
dupes = base.duplicated(["raceId", "driverId"]).sum()
assert dupes == 0, "more than one row for a driver in a race"
assert set(base["podium"].unique()) <= {0, 1}, "podium is not binary"

print("duplicated (raceId, driverId):", dupes)
print("target values:", sorted(base["podium"].unique()))
print("overall podium rate:", round(base["podium"].mean(), 4))

In [ ]:
import matplotlib.pyplot as plt

# Missing share (%) per year for the qualifying columns
qual_cols = ["qual_position", "q1_s", "q2_s", "q3_s"]
miss_by_year = base.groupby("year")[qual_cols].apply(lambda d: d.isna().mean()) * 100

fig, ax = plt.subplots(figsize=(16, 3.5))
im = ax.imshow(miss_by_year.T, aspect="auto", cmap="Blues", vmin=0, vmax=100)

# Axis labels: columns on y, every 5th year on x
years = list(miss_by_year.index)
ax.set_yticks(range(len(qual_cols)), qual_cols)
ax.set_xticks(range(0, len(years), 5), years[::5])

# Mark the 2006 start year
ax.axvline(years.index(2006) - 0.5, color="red", linestyle="--", linewidth=2)
ax.text(years.index(2006), -0.7, "2006 (knockout starts)", color="red", fontsize=9)

fig.colorbar(im, ax=ax, label="% of rows missing")
ax.set_title("Missing qualifying data per year (dark = more missing)", pad=18)
ax.set_xlabel("Year")
plt.tight_layout()
plt.savefig("../reports/fig_p3_qual_missing_per_year.png", dpi=120, bbox_inches="tight")
plt.show()

### Section 3.3: Training start year

The ml table uses qualifying features (qual_position, q1_s, q2_s, q3_s,
gap_to_pole_pct). 

Qualifying data does not exist for most of F1 history (e.g. 1950-1993),
so we need to choose the earliest stable year to include in the ML table.

We look at qualifying coverage per season and pick the earliest year where it's stable.

In [ ]:
# Calendar table needed to bring year/round into standings and results
races = tables["races"][["raceId", "year", "round", "circuitId", "date"]].copy()
races.head()

In [ ]:
# Find the first year with any qualifying data and the first year with high coverage
q_cov_exact = base.groupby("year")["has_qualifying"].mean()

# Years before this have no qualifying data at all
years_with_any = q_cov_exact[q_cov_exact > 0].index.min()
# Years before this have full or near-full coverage
years_with_full = q_cov_exact[q_cov_exact >= 0.90].index.min()

print("=" * 60)
print("First year with ANY qualifying data (has_qualifying > 0):", years_with_any)
print("First year with >=90% coverage:                      ", years_with_full)
print("=" * 60)

print("\nCoverage 1990–1997 (the transition years):")
print(q_cov_exact.loc[1990:1997].round(4))

In [ ]:
# Qualifying coverage per season: share of rows with a q1_s time
cov = base.groupby("year")["has_qualifying"].mean()

# Also look at q1_s, q2_s, q3_s separately
q_cov = base.groupby("year")[["q1_s", "q2_s", "q3_s"]].apply(lambda d: d.notna().mean())

print("has_qualifying per year (first 10 and last 10):")
print(cov.head(10))
print(cov.tail(10))

print("\nq1_s / q2_s / q3_s coverage for 1993-2007:")
print(q_cov.loc[1993:2007])

In [ ]:
print("\nq1_s / q2_s / q3_s coverage for 2006-2024:")
print(q_cov.loc[2006:2024])

**From 2006 the qualifying data is stable for q1, q2 and q3, because the knockout format starts in 2006. Earlier seasons have little or no qualifying data, so they are kept but get a lower weight in training (see below).**

In [ ]:
START_YEAR = 1950          # keep every season
FULL_WEIGHT_FROM = 2006    # knockout qualifying: all qualifying columns exist from here
OLD_SEASON_WEIGHT = 0.25   # pre-2006 rows count a quarter in training (tested in 3.11, Part F)
model = base[base["year"] >= START_YEAR].copy()
model = model.sort_values(["year", "round", "raceId", "driverId"]).reset_index(drop=True)

print("Rows kept:", model.shape[0])
print("Years:", model["year"].min(), "-", model["year"].max())
print("Rows dropped:", base.shape[0] - model.shape[0])
print("\nmodel shape:", model.shape)
print("model years:", model["year"].min(), "-", model["year"].max())

In [ ]:
import matplotlib.pyplot as plt

# Coverage per season for each qualifying time column
q_cov = (
    base.groupby("year")[["q1_s", "q2_s", "q3_s"]]
        .apply(lambda d: d.notna().mean())
)

# Focus on the era where qualifying data actually exists
q_cov = q_cov.loc[1994:]

fig, ax = plt.subplots(figsize=(12, 5))

for col, color in zip(["q1_s", "q2_s", "q3_s"],
                      ["tab:blue", "tab:orange", "tab:green"]):
    ax.plot(q_cov.index, q_cov[col], marker="o", label=col, color=color)

ax.axvline(2006, color="red", linestyle="--", label="full weight from 2006")

ax.set_title("Qualifying time coverage per season, 1994–2024")
ax.set_xlabel("Year")
ax.set_ylabel("Share of rows with a time")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../reports/fig_p3_qual_coverage_per_year.png",
            dpi=120, bbox_inches="tight")
plt.show()

**Why we keep every season, with a lower weight before 2006**

The plot shows qualifying coverage per season. Before 1994 there is no qualifying data, 1994–95 have near-full q1 coverage, 1996–2002 are patchy, 2003–2005 have q1 only, and from 2006 (knockout format) all three sessions are stable.

| Column | Range 2006–2024 | Meaning |
|---|---|---|
| `q1_s` | 0.97 – 1.00 | Almost every entrant sets a Q1 time |
| `q2_s` | 0.70 – 0.75 | ~15 of ~20 drivers reach Q2 |
| `q3_s` | 0.37 – 0.50 | ~10 of ~20 drivers reach Q3 |

The lower plateaus are structural, not missing data: only the top 15 reach Q2 and the top 10 reach Q3.

**Decision.** We keep every season from 1950 (24,566 rows) instead of dropping 1950–2005. In training, rows from 2006 on get weight 1 and rows before 2006 get weight `OLD_SEASON_WEIGHT = 0.25`, stored in the `sample_weight` column. The old seasons count less because most of them have no qualifying data and come from a different version of the sport (more cars per race, more retirements, other points systems). Section 3.11, Part F compares this with dropping the old seasons and with using them at full weight.

### Section 3.4: Time split

We split by year. The model's job is to predict
future races from pre-race information, so training on later rows and
testing on earlier ones would leak future information into the model. A
time split matches how the model will actually be used.

- **Train**: year ≤ 2019 (22,441 rows: 16,699 from 1950–2005 with weight 0.25, 5,742 from 2006–2019 with weight 1)
- **Validation**: 2020–2021 (774 rows)
- **Test**: year ≥ 2022 (1,351 rows)

**Why these boundaries.**
Training covers every season up to 2019 (pre-COVID), with the seasons before 2006 down-weighted. Validation covers the nearest seasons to the test set and is used for model, threshold and preprocessing choices. Its weakness is that 2020 was shortened by COVID, so we report results per season. Test covers the ground-effect era, a regulation change the model never sees in training, and is used once.

**Weights.** Only training rows before 2006 have a weight below 1. Validation and test rows all have weight 1, so the scores are not affected by the weights.

**Class balance.** Train 13.5% (slightly lower because early races had more cars), validation 15.1%, test 15.1%.

**Alternatives.** A random split leaks same-season information. Random assignment of seasons mixes regulations.

In [ ]:
import numpy as np

model["split"] = np.select(
    [model["year"] <= 2019,
     model["year"].between(2020, 2021),
     model["year"] >= 2022],
    ["train", "validation", "test"],
    default="unknown"
)

# Training weight: seasons before 2006 count less, because most of them have
# little or no qualifying data. Validation and test rows (2020+) all have weight 1.
model["sample_weight"] = np.where(model["year"] < FULL_WEIGHT_FROM, OLD_SEASON_WEIGHT, 1.0)

print(model["split"].value_counts())
print("\nRows and weight per split:")
print(model.groupby("split")["sample_weight"].agg(rows="count", weighted_rows="sum", mean_weight="mean").round(2))
print("\nmodel shape:", model.shape)

In [ ]:
# Class balance per split
balance = model.groupby("split")["podium"].agg(["mean", "count"])
print("Class balance per split:")
print(balance.round(4))

# Plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
balance_ordered = balance.reindex(["train", "validation", "test"])
bars = ax.bar(balance_ordered.index, balance_ordered["mean"] * 100,
              color=["tab:blue", "tab:orange", "tab:green"])
ax.bar_label(bars, fmt="%.1f%%")
ax.set_title("Podium rate per split (class balance)")
ax.set_xlabel("Split")
ax.set_ylabel("Podium rate (%)")
ax.set_ylim(0, 20)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("../reports/fig_p3_split_balance.png", dpi=120, bbox_inches="tight")
plt.show()

### Section 3.5: Grid and qualifying features

The prediction cutoff is after qualifying and before the race. Every feature
below is set during or before grid formation; none comes from the race itself.

**Grid features**

- `grid_fixed` is the ordinal starting position. The raw `grid` column uses
  `0` for a pit-lane start, which a linear model would read as ahead of pole. We set those to PIT_GRID = 35, one place past the largest grid in the training years (34, in 1953). The value is derived from training data only, and a range check confirms no validation or test grid exceeds the training maximum.
- pit_lane_flag is 1 for pit-lane starts and 0 otherwise. It keeps the start type that grid_fixed compresses into one ordinal value.

**Qualifying features**

- `qual_position` is the driver's rank in the qualifying session.
- `best_qual_s` is the driver's best qualifying lap in seconds
  (`min(q1_s, q2_s, q3_s)` across the sessions they reached).
- `gap_to_pole_pct` is the percentage by which the driver’s best time is slower than the fastest time in the same race. Pole is defined by fastest time, not by grid position, because 64 races have a pole starter who was not the fastest qualifier. 61 of them are from 2006 on, and 39 of those from 2006–2009, when Q3 was run on race fuel: the cars were heavier in Q3, so the fastest lap of the day often came from Q2.
- `no_qual_time` is 1 when the driver set no qualifying time at all
  (`best_qual_s` is NaN). There is no qualifying data before 1994, so this
  is 1 for 14,306 rows (58%), almost all before 2006. From 2006 on it is 1
  for only 96 rows (21 of them also have no `qual_position`): drivers
  excluded from qualifying, withdrawn, or unrecorded, who start from the
  back or the pit lane.
- `qual_group` is the last session the driver reached (Q3 / Q2 only /
  Q1 only / no qualifying time), with yes/no columns `qg_q3`,
  `qg_q2_only` and `qg_q1_only`. The model uses only `qg_q3` and
  `qg_q2_only`: since `qg_q3 + qg_q2_only + qg_q1_only + no_qual_time = 1`
  on every row, `qg_q1_only` can be derived from the others and adds nothing. It only has its full meaning
  in the knockout format (2006+).

  **Why `q1_s`, `q2_s`, `q3_s` are not features.** The raw session times
are used to build `best_qual_s` (the minimum across sessions the driver
reached) and `gap_to_pole_pct` (the normalised gap to the fastest lap in
the race). The raw columns are not in `FEATURES` for three reasons:

1. Their missingness is structural: a Q1-eliminated driver has NaN Q2 and
   Q3. `best_qual_s` and `gap_to_pole_pct` handle that by falling back on
   whichever sessions the driver reached.
2. Absolute lap times vary by track, so the
   raw seconds are not comparable across races. `gap_to_pole_pct`
   normalises by the race's fastest time.
3. Including all three would add three highly correlated columns with
   different missingness patterns for the same underlying fact.

In [ ]:
# Grid features: grid_fixed (ordinal) + pit_lane_flag (indicator)

# 1. Max real grid slot, from TRAIN ONLY.
max_grid = model.loc[
    (model["split"] == "train") & (model["grid"] > 0), "grid"
].max()
print("Max real grid slot (train years only):", max_grid)

# 2. Sentinel: one past the largest real grid slot.
PIT_GRID = max_grid + 1
print(f"PIT_GRID = {PIT_GRID}")

# 3. Build the two features.
model["pit_lane_flag"] = (model["grid"] == 0).astype(int)
model["grid_fixed"] = model["grid"].where(model["grid"] > 0, PIT_GRID)

# 4. Crosstab: pit_lane_flag agrees with pitlane_start.
#    pitlane_start is defined as grid == 0 in f1_clean, so this always
#    agrees (consistency check only).
print("\nCrosstab (grid==0 vs pitlane_start):")
print(pd.crosstab(model["grid"] == 0, model["pitlane_start"]))

# 5. Distribution tail.
print("\nGrid value counts (last 5):")
print(model["grid_fixed"].value_counts().sort_index().tail())

# 6. Assertions.
assert model["grid_fixed"].notna().all(), "null grid after recode"
assert model["grid_fixed"].between(1, PIT_GRID).all()
assert (model.loc[model["pit_lane_flag"] == 1, "grid_fixed"] == PIT_GRID).all()

# 7. Range check: no val/test grid exceeds the train max.
leak_check = model.loc[
    (model["split"] != "train") & (model["grid"] > max_grid)
]
assert len(leak_check) == 0, \
    f"val/test rows with real grid > train max_grid: {len(leak_check)}"

print(f"\nPit lane starts: {model['pit_lane_flag'].sum()} rows")
print(f"Rows at sentinel ({PIT_GRID}): {(model['grid_fixed'] == PIT_GRID).sum()}")
print(f"Val/test rows with grid above train max ({max_grid}): {len(leak_check)}")
print("Grid features: OK")

In [ ]:
# Qualifying features
model["best_qual_s"] = model[["q1_s", "q2_s", "q3_s"]].min(axis=1)

# Gap to pole (percentage)
pole = model.groupby("raceId")["best_qual_s"].transform("min")
model["gap_to_pole_pct"] = (model["best_qual_s"] - pole) / pole * 100

# No-time flag: 1 when the driver set no qualifying time at all
model["no_qual_time"] = model["gap_to_pole_pct"].isna().astype(int)

print(model[[
    "grid_fixed", "pit_lane_flag", "qual_position",
    "best_qual_s", "gap_to_pole_pct", "no_qual_time",
]].describe())

In [ ]:
defined = model["gap_to_pole_pct"].notna()
assert (model.loc[defined, "gap_to_pole_pct"] >= 0).all(), \
    "negative gap found on rows with a defined gap"

# Pole = fastest best_qual_s in the race, not qual_position == 1
model["is_pole_time"] = (
    model["best_qual_s"]
    == model.groupby("raceId")["best_qual_s"].transform("min")
)
assert (model.loc[model["is_pole_time"] & defined, "gap_to_pole_pct"] == 0).all(), \
    "driver with race-minimum best_qual_s does not have gap 0"

print(f"Rows with defined gap: {defined.sum()}")
print(f"Rows with NaN gap (no qualifying time at all): {(~defined).sum()}")
print(f"Rows with pole-time: {model['is_pole_time'].sum()}")

# Optional: how often qual_position == 1 is not the fastest time
mismatch = model[(model["qual_position"] == 1) & ~model["is_pole_time"]]
print(f"Races where qual_position == 1 is not the fastest qualifier: "
      f"{mismatch['raceId'].nunique()}")

# Confirm the NaN gap is fully explained by missing qualifying data
all_q_missing = model[["q1_s", "q2_s", "q3_s"]].isna().all(axis=1)
assert (all_q_missing == model["gap_to_pole_pct"].isna()).all(), \
    "NaN gap does not match all-three-missing"
print("NaN gap == all three qualifying times missing: OK")

In [ ]:
import numpy as np

model["qual_group"] = np.select(
    [model["q3_s"].notna(),
     model["q2_s"].notna(),
     model["q1_s"].notna()],
    ["Q3", "Q2 only", "Q1 only"],
    default="no qualifying time",
)

# Yes/no columns for the table.
for label, col in zip(["Q3", "Q2 only", "Q1 only"], ["qg_q3", "qg_q2_only", "qg_q1_only"]):
    model[col] = (model["qual_group"] == label).astype(int)

# The model uses two of them: qg_q1_only = 1 - qg_q3 - qg_q2_only - no_qual_time,
# so it carries no extra information ("Q1 only" is the case where all three are 0).
assert (model[["qg_q3", "qg_q2_only", "qg_q1_only", "no_qual_time"]].sum(axis=1) == 1).all()
QUAL_GROUP_COLS = ["qg_q3", "qg_q2_only"]

print(
    model.groupby("qual_group", observed=True)
         .agg(
             n_rows=("gap_to_pole_pct", "size"),
             n_defined=("gap_to_pole_pct", "count"),
             mean=("gap_to_pole_pct", "mean"),
             median=("gap_to_pole_pct", "median"),
             max=("gap_to_pole_pct", "max"),
         )
         .round(3)
)

In [ ]:
# Candidate: gap to the fastest Q1 lap in the same race (%).
# Every driver runs Q1, and Q1 was never run on race fuel, so all laps are
# comparable. Tested against gap_to_pole_pct in Section 3.11.
q1_pole = model.groupby("raceId")["q1_s"].transform("min")
model["gap_to_pole_q1_pct"] = (model["q1_s"] - q1_pole) / q1_pole * 100

defined_q1 = model["gap_to_pole_q1_pct"].notna()
assert (model.loc[defined_q1, "gap_to_pole_q1_pct"] >= 0).all(), "negative Q1 gap"
print(model["gap_to_pole_q1_pct"].describe().round(3))
print("missing share:", round(model["gap_to_pole_q1_pct"].isna().mean(), 4))

### Section 3.6: Pre-race standings

The standings tables record cumulative points and rank **after** each
race. Using them directly would leak the current race's result into the
features.

Fix: sort the standings by `(year, round)`, then `shift(1)` within
`(year, driverId)` and `(year, constructorId)` so each row holds the
standings going into that race. The shift never uses the current race.

Round 1 has no previous race: points are 0 (the true value), position is
undefined (NaN), flagged by `standings_missing`. Coverage gaps — the 419
`(raceId, driverId)` pairs inside the modeling window where the driver
raced but has no standings row — are also filled with 0 for points and
flagged with `standings_missing = 1`. A diagnostic confirmed none of them
has an earlier standing in the same season, so 0 is the correct value, not
a lost earlier total. The same holds for the 399 `(raceId, constructorId)`
gaps, most of them before 1958, when there was no constructors' championship.

We assert: one standings row per `(year, id, round)`, round-1 points
are 0, round-1 position is NaN.

In [ ]:
ds = tables["driver_standings"][
    ["raceId", "driverId", "points", "position"]
].copy()
ds = ds.merge(races, on="raceId", how="left")
ds = ds.sort_values(["year", "round", "driverId"])

assert not ds.duplicated(["year", "driverId", "round"]).any(), \
    "duplicate driver standings row"

# Shift within (year, driverId): round N gets round N-1's values.
ds["pts_before"] = ds.groupby(["year", "driverId"])["points"].shift(1)
ds["pos_before"] = ds.groupby(["year", "driverId"])["position"].shift(1)

# Round 1 has no previous race: points are 0, position is undefined.
ds["pts_before"] = ds["pts_before"].fillna(0.0)
ds["standings_missing"] = ds["pos_before"].isna().astype("int8")

ds_pre = ds[["raceId", "driverId", "pts_before", "pos_before",
             "standings_missing"]]

print("ds_pre shape:", ds_pre.shape)

In [ ]:
# Safe to rerun: drop existing standings columns first.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["pts_before", "pos_before", "standings_missing",
                      "pts_before_x", "pts_before_y",
                      "pos_before_x", "pos_before_y",
                      "standings_missing_x", "standings_missing_y"]],
    errors="ignore",
)

model = model.merge(ds_pre, on=["raceId", "driverId"], how="left", validate="many_to_one")
model["pts_before"] = model["pts_before"].fillna(0.0)

model["standings_missing"] = model["standings_missing"].fillna(1).astype("int8")   # 🆕
assert model["standings_missing"].notna().all(), "standings flag has empty values"  # 🆕

first_race = model.groupby("year")["round"].transform("min") == model["round"]
assert model.loc[first_race, "pts_before"].eq(0.0).all(), "round 1 should have 0 points"
assert model.loc[first_race, "pos_before"].isna().all(), "round 1 position should be NaN"
print("Driver standings: OK")

In [ ]:
cs = tables["constructor_standings"][
    ["raceId", "constructorId", "points", "position"]
].copy()
cs = cs.merge(races, on="raceId", how="left")
cs = cs.sort_values(["year", "round", "constructorId"])

assert not cs.duplicated(["year", "constructorId", "round"]).any(), \
    "duplicate constructor standings row"

cs["cons_pts_before"] = cs.groupby(["year", "constructorId"])["points"].shift(1)
cs["cons_pos_before"] = cs.groupby(["year", "constructorId"])["position"].shift(1)

cs["cons_pts_before"] = cs["cons_pts_before"].fillna(0.0)
cs["cons_standings_missing"] = cs["cons_pos_before"].isna().astype("int8")

cs_pre = cs[["raceId", "constructorId",
             "cons_pts_before", "cons_pos_before",
             "cons_standings_missing"]]

print("cs_pre shape:", cs_pre.shape)

In [ ]:
# Safe to rerun: drop existing standings columns first.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["cons_pts_before", "cons_pos_before",
                      "cons_standings_missing",
                      "cons_pts_before_x", "cons_pts_before_y",
                      "cons_pos_before_x", "cons_pos_before_y",
                      "cons_standings_missing_x", "cons_standings_missing_y"]],
    errors="ignore",
)

model = model.merge(cs_pre, on=["raceId", "constructorId"],
                    how="left", validate="many_to_one")
model["cons_pts_before"] = model["cons_pts_before"].fillna(0.0)
model["cons_standings_missing"] = model["cons_standings_missing"].fillna(1).astype("int8")   # 🆕
assert model["cons_standings_missing"].notna().all(), "constructor flag has empty values"   # 🆕


assert model.loc[first_race, "cons_pts_before"].eq(0.0).all(), \
    "round 1 should have 0 constructor points"
assert model.loc[first_race, "cons_pos_before"].isna().all(), \
    "round 1 constructor position should be NaN"
print("Constructor standings: OK")

print("model shape:", model.shape)
print(model[["pts_before", "pos_before", "standings_missing",
             "cons_pts_before", "cons_pos_before",
             "cons_standings_missing"]].isna().mean())

In [ ]:
# Points share of the leader going into the race (0 to 1).
# Raw points are not comparable across years: a win was worth 10 points before
# 2010 and 25 after. The share is. Candidate feature, tested in the ablation (3.11).
leader = model.groupby("raceId")["pts_before"].transform("max")
model["pts_share"] = (model["pts_before"] / leader).where(leader > 0, 0.0)

cons_leader = model.groupby("raceId")["cons_pts_before"].transform("max")
model["cons_pts_share"] = (model["cons_pts_before"] / cons_leader).where(cons_leader > 0, 0.0)

assert model[["pts_share", "cons_pts_share"]].notna().all().all(), "empty share values"
assert model["pts_share"].between(0, 1).all() and model["cons_pts_share"].between(0, 1).all()
print(model[["pts_share", "cons_pts_share"]].describe().round(3))

**Recent form (candidate features).** `driver_form5` is the driver's podium
rate over their previous 5 races; `constructor_form5` is the same for the
team (per race, the share of its drivers on the podium). Both are computed on
all years, so an early-2006 row still sees the end of 2005, and both use
`shift(1)` before the rolling window, so a row never sees its own race. For
constructors we first collapse to one row per team-race, so a teammate's
result in the same race cannot leak in. A driver's or team's first race has
no history and stays NaN. Whether form adds anything beyond the standings is
tested in Section 3.11.

In [ ]:
# Safe to rerun.
model = model.drop(columns=["driver_form5", "constructor_form5"], errors="ignore")

# Driver form: podium rate over the previous 5 races (all years).
form_d = base.sort_values(["driverId", "year", "round"]).reset_index(drop=True)
form_d["driver_form5"] = (
    form_d.groupby("driverId")["podium"]
          .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
)

# Constructor form: one row per team-race first, then the same window.
form_c = (
    base.groupby(["constructorId", "raceId", "year", "round"], as_index=False)["podium"].mean()
        .sort_values(["constructorId", "year", "round"])
        .reset_index(drop=True)
)
form_c["constructor_form5"] = (
    form_c.groupby("constructorId")["podium"]
          .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
)

model = model.merge(form_d[["raceId", "driverId", "driver_form5"]],
                    on=["raceId", "driverId"], how="left", validate="one_to_one")
model = model.merge(form_c[["raceId", "constructorId", "constructor_form5"]],
                    on=["raceId", "constructorId"], how="left", validate="many_to_one")

# Leakage checks: no history on a first race, teammates share the team value.
first_d = form_d.groupby("driverId").cumcount() == 0
assert form_d.loc[first_d, "driver_form5"].isna().all(), "driver form on a first career race"
first_c = form_c.groupby("constructorId").cumcount() == 0
assert form_c.loc[first_c, "constructor_form5"].isna().all(), "constructor form on a first race"
assert (model.groupby(["raceId", "constructorId"])["constructor_form5"]
             .nunique(dropna=False) <= 1).all(), "teammates have different constructor form"

print(model[["driver_form5", "constructor_form5"]].describe().round(3))
print("missing share:", model[["driver_form5", "constructor_form5"]].isna().mean().round(4).to_dict())

### Section 3.7: Circuit history + home flag + season progress + era ###

We also need to add these to our features:

**Circuit history.** Some drivers are specialists at certain tracks (e.g. Senna
at Monaco, Hamilton at Silverstone). We build five circuit features:
- `circuit_starts_before`: number of previous races by this driver at this circuit
- `circuit_podiums_before`: number of previous podiums by this driver at this circuit
- `circuit_podium_rate`: the driver's smoothed podium rate at this circuit,
  `(podiums + PRIOR × 5) / (starts + 5)`. A driver with few visits is pulled
  toward the average instead of showing 0% or 100%.
- `team_circuit_starts_before`: number of previous races by this constructor at this circuit
- `team_circuit_podiums_before`: the constructor's previous podiums at this circuit
  (per race, the share of its drivers on the podium, so one of two drivers = 0.5)

All five use only past races. The current race is excluded by `cumcount()` and
`shift(1).cumsum()`. They are computed on the full `base` table (all years), so a
2006 row still knows the driver's earlier visits.

**Home flag.** `home_flag = 1` when the driver's nationality matches the
circuit's country (e.g. British driver at Silverstone).

**Season progress.** `season_progress = round / last round of the season`
(0 to 1). The calendar is known before the season starts.

**Era.** F1 rules and car performance shift by technical era. We bucket
years: `pre_2006` (1950–2005), `early_v8` (2006–2008), `late_v8` (2009–2013),
`hybrid` (2014–2021), `ground_effect` (2022+), and encode them as `era_ordinal` (0 to 4).

**Note on `era_ordinal`.** It is in `FEATURES` only as a candidate for the
ablation (Section 3.11). It has a structural limitation: the ground-effect era
(2022+, value 4) never appears in training (1950–2019), and validation
(2020–2021) is entirely hybrid, so the effect of value 3 cannot be learned or
checked before the test set. The final decision is made in Section 3.11.

In [ ]:
base_sorted_circuit = (
    base.sort_values(["driverId", "circuitId", "year", "round"])
        .reset_index(drop=True)
)

base_sorted_circuit["circuit_starts_before"] = (
    base_sorted_circuit.groupby(["driverId", "circuitId"]).cumcount()
)
base_sorted_circuit["circuit_podiums_before"] = (
    base_sorted_circuit.groupby(["driverId", "circuitId"])["podium"]
                       .transform(lambda s: s.shift(1).fillna(0).cumsum())
)

# Safe to rerun.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["circuit_starts_before", "circuit_podiums_before",
                      "circuit_podium_rate",
                      "circuit_starts_before_x", "circuit_starts_before_y",
                      "circuit_podiums_before_x", "circuit_podiums_before_y"]],
    errors="ignore",
)

model = model.merge(
    base_sorted_circuit[["raceId", "driverId",
                         "circuit_starts_before",
                         "circuit_podiums_before"]],
    on=["raceId", "driverId"],
    how="left",
)

# Smoothed podium rate.
PRIOR = 0.15
PRIOR_STRENGTH = 5
model["circuit_podium_rate"] = (
    (model["circuit_podiums_before"] + PRIOR * PRIOR_STRENGTH)
    / (model["circuit_starts_before"] + PRIOR_STRENGTH)
)

print(model[["circuit_starts_before", "circuit_podiums_before",
             "circuit_podium_rate"]].describe().round(3))

In [ ]:
first_at_circuit = (
    base_sorted_circuit.sort_values(["driverId", "circuitId", "year", "round"])
                      .groupby(["driverId", "circuitId"])
                      .cumcount() == 0
)
assert (base_sorted_circuit.loc[first_at_circuit, "circuit_starts_before"] == 0).all()
assert (base_sorted_circuit.loc[first_at_circuit, "circuit_podiums_before"] == 0).all()
print("Circuit history leakage check: OK")

In [ ]:
base_team_circuit = (
    base
      .groupby(["constructorId", "circuitId", "raceId", "year", "round"],
               as_index=False)["podium"]
      .mean()
)
base_team_circuit = base_team_circuit.sort_values(
    ["constructorId", "circuitId", "year", "round"]
).reset_index(drop=True)

base_team_circuit["team_circuit_starts_before"] = (
    base_team_circuit.groupby(["constructorId", "circuitId"]).cumcount()
)
base_team_circuit["team_circuit_podiums_before"] = (
    base_team_circuit.groupby(["constructorId", "circuitId"])["podium"]
                    .transform(lambda s: s.shift(1).fillna(0).cumsum())
)

# Safe to rerun.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["team_circuit_starts_before",
                      "team_circuit_podiums_before",
                      "team_circuit_starts_before_x",
                      "team_circuit_starts_before_y",
                      "team_circuit_podiums_before_x",
                      "team_circuit_podiums_before_y"]],
    errors="ignore",
)

model = model.merge(
    base_team_circuit[["raceId", "constructorId",
                       "team_circuit_starts_before",
                       "team_circuit_podiums_before"]],
    on=["raceId", "constructorId"],
    how="left",
    validate="many_to_one",
)

print(model[["team_circuit_starts_before",
             "team_circuit_podiums_before"]].describe().round(3))

In [ ]:
NATIONALITY_TO_COUNTRY = {
    "American": "USA", "American-Italian": "USA", "Argentine": "Argentina",
    "Argentine-Italian": "Argentina", "Argentinian": "Argentina",
    "Australian": "Australia", "Austrian": "Austria", "Belgian": "Belgium",
    "Brazilian": "Brazil", "British": "UK", "Canadian": "Canada",
    "Chilean": "Chile", "Chinese": "China", "Colombian": "Colombia",
    "Czech": "Czech Republic", "Danish": "Denmark", "Dutch": "Netherlands",
    "East German": "Germany", "Finnish": "Finland", "French": "France",
    "German": "Germany", "Hungarian": "Hungary", "Indian": "India",
    "Indonesian": "Indonesia", "Irish": "Ireland", "Italian": "Italy",
    "Japanese": "Japan", "Liechtensteiner": "Liechtenstein",
    "Malaysian": "Malaysia", "Mexican": "Mexico", "Monegasque": "Monaco",
    "New Zealander": "New Zealand", "Polish": "Poland",
    "Portuguese": "Portugal", "Rhodesian": "Rhodesia", "Russian": "Russia",
    "South African": "South Africa", "Spanish": "Spain", "Swedish": "Sweden",
    "Swiss": "Switzerland", "Thai": "Thailand", "Uruguayan": "Uruguay",
    "Venezuelan": "Venezuela",
}
SECOND_COUNTRY = {"American-Italian": "Italy", "Argentine-Italian": "Italy"}

drivers_nat = tables["drivers"][["driverId", "nationality"]].copy()
drivers_nat["nationality"] = drivers_nat["nationality"].str.strip()

circuits_country = tables["circuits"][["circuitId", "country"]].copy()
circuits_country["country"] = circuits_country["country"].replace(
    "United States", "USA"
)

# Safe to rerun.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["nationality", "country", "home_country",
                      "second_home_country", "home_flag",
                      "nationality_x", "nationality_y",
                      "country_x", "country_y"]],
    errors="ignore",
)

model = model.merge(drivers_nat, on="driverId", how="left", validate="many_to_one")
model = model.merge(circuits_country, on="circuitId", how="left", validate="many_to_one")

model["home_country"] = model["nationality"].map(NATIONALITY_TO_COUNTRY)
model["second_home_country"] = model["nationality"].map(SECOND_COUNTRY)
model["home_flag"] = (
    (model["country"] == model["home_country"]) |
    (model["country"] == model["second_home_country"])
).astype(int)

assert model["nationality"].notna().all()
assert model["country"].notna().all()
unmapped = set(model["nationality"]) - set(NATIONALITY_TO_COUNTRY)
assert len(unmapped) == 0, f"unmapped nationalities: {unmapped}"

# Sanity check: British drivers at Silverstone (circuit 9).
silverstone = model[(model["circuitId"] == 9) & (model["home_flag"] == 1)]
print("UK drivers flagged at Silverstone:", len(silverstone))
assert len(silverstone) > 0, "home_flag broken: no UK drivers flagged at Silverstone"
print(model["home_flag"].value_counts())

model = model.drop(
    columns=["nationality", "country", "home_country", "second_home_country"],
    errors="ignore",
)

In [ ]:
# Season progress: fraction of the season completed before this race.
season_length = model.groupby("year")["round"].transform("max")
model["season_progress"] = model["round"] / season_length

print(model["season_progress"].describe().round(3))

In [ ]:
model["era"] = pd.cut(
    model["year"],
    bins=[1949, 2005, 2008, 2013, 2021, 2024],
    labels=["pre_2006", "early_v8", "late_v8", "hybrid", "ground_effect"],
)

# Ordinal encoding so the column is numeric and usable by the pipeline.
# The model treats eras as ordered in time; this is an approximation, see
# the markdown note.
ERA_ORDINAL = {"pre_2006": 0, "early_v8": 1, "late_v8": 2, "hybrid": 3, "ground_effect": 4}
model["era_ordinal"] = model["era"].map(ERA_ORDINAL).astype("Int64")

print(model["era"].value_counts())
print("\nera and era_ordinal mapping:")
print(model[["era", "era_ordinal"]].drop_duplicates().sort_values("era_ordinal"))

assert model["era_ordinal"].notna().all(), "era mapping missed a category"
print("\nmodel shape:", model.shape)

In [ ]:
print("Columns in model:")
for i, col in enumerate(sorted(model.columns), 1):
    print(f"{i:2d}. {col}")
print("\nTotal:", len(model.columns))

### Section 3.8: Data dictionary and feature groups

We keep two lists.

**`CANDIDATE_GROUPS`: the 20 candidate features** of the first version, in six groups. Section 3.11 tests them.

- **grid**: where the car starts.
- **qualifying**: how fast the driver was in qualifying.
- **driver_standings**: the driver's championship points and position going into the race.
- **constructor_standings**: the team's championship points and position going into the race.
- **circuit**: the driver's and team's history at this specific track.
- **context**: home race, era, and season progress.

**`FEATURE_GROUPS`: the final 12 features** (Option 1 without the redundant `qg_q1_only`), chosen by the ablation in Section 3.11:

- **grid**: `grid_fixed`
- **qualifying**: `qual_position`, `gap_to_pole_pct`, `no_qual_time`, `qg_q3`, `qg_q2_only`
- **driver_standings**: `pts_before`, `pos_before`, `standings_missing`
- **constructor_standings**: `cons_pts_before`, `cons_pos_before`, `cons_standings_missing`

Identifiers (`driverId`, `constructorId`, `circuitId`, `number`) and
keys (`raceId`, `year`, `round`, `date`, `split`) are excluded from
`FEATURES`. They are useful for joins and audit, not as model inputs: an ID
is a label, not a quantity (driver 830 is not "more" than driver 815), so a
model would learn false patterns from the numbers.

`sample_weight` is a training weight, not a feature: it is passed to the
model's `fit` (pre-2006 rows count 0.25).

**Saved for the model notebooks:** `p3_feature_groups.json` (final list),
`p3_data_dictionary.csv`, `p3_training_config.json` (start year and weights),
and `p3_modeling_table.csv` (all columns, including `sample_weight`).

In [ ]:
# The 20 candidate features of the first version. Section 3.11 tests them.
CANDIDATE_GROUPS = {
    "grid":                  ["grid_fixed", "pit_lane_flag"],
    "qualifying":            ["qual_position", "best_qual_s", "gap_to_pole_pct", "no_qual_time"],
    "driver_standings":      ["pts_before", "pos_before", "standings_missing"],
    "constructor_standings": ["cons_pts_before", "cons_pos_before", "cons_standings_missing"],
    "circuit":               ["circuit_starts_before", "circuit_podiums_before", "circuit_podium_rate",
                              "team_circuit_starts_before", "team_circuit_podiums_before"],
    "context":               ["home_flag", "era_ordinal", "season_progress"],
}
CANDIDATES = [f for cols in CANDIDATE_GROUPS.values() for f in cols]

# Final feature list (Option 1, chosen by the ablation in Section 3.11).
FEATURE_GROUPS = {
    "grid":                  ["grid_fixed"],
    "qualifying":            ["qual_position", "gap_to_pole_pct", "no_qual_time"] + QUAL_GROUP_COLS,
    "driver_standings":      ["pts_before", "pos_before", "standings_missing"],
    "constructor_standings": ["cons_pts_before", "cons_pos_before", "cons_standings_missing"],
}
FEATURES = [f for group in FEATURE_GROUPS.values() for f in group]

missing = [f for f in CANDIDATES + FEATURES if f not in model.columns]
assert not missing, f"feature lists reference missing columns: {missing}"
assert "sample_weight" not in FEATURES, "the training weight is not a feature"

print(f"Candidates: {len(CANDIDATES)} features in {len(CANDIDATE_GROUPS)} groups")
print(f"Final:      {len(FEATURES)} features in {len(FEATURE_GROUPS)} groups")
for group, cols in FEATURE_GROUPS.items():
    print(f"  {group}: {cols}")

In [ ]:
FORBIDDEN = {"positionOrder", "position", "positionText", "points", "laps",
             "time", "milliseconds", "fastestLap", "rank", "status"}
assert not (set(FEATURES) & FORBIDDEN), \
    f"forbidden in FEATURES: {set(FEATURES) & FORBIDDEN}"
print("All features are leakage-safe.")

In [ ]:
import pandas as pd
from IPython.display import display, HTML

FEATURE_SOURCE = {
    "grid_fixed": ("grid", f"Starting slot after penalties; pit-lane starts set to PIT_GRID={PIT_GRID}."),
    "pit_lane_flag": ("grid", "1 if the driver started from the pit lane."),
    "qual_position": ("qualifying", "Rank in the qualifying session."),
    "best_qual_s": ("qualifying", "Best qualifying lap across the sessions reached."),
    "gap_to_pole_pct": ("qualifying", "Percent slower than the fastest lap in the same race."),
    "no_qual_time": ("qualifying", "1 if the driver set no qualifying time at all."),
    "qg_q3": ("qualifying", "1 if the driver reached Q3 (set a Q3 time)."),
    "qg_q2_only": ("qualifying", "1 if the driver's last session was Q2."),
    "qg_q1_only": ("qualifying", "1 if the driver's last session was Q1."),
    "pts_before": ("driver_standings", "Driver points going into the race."),
    "pos_before": ("driver_standings", "Driver championship position going into the race."),
    "standings_missing": ("driver_standings", "1 if the driver has no previous standings row this season."),
    "cons_pts_before": ("constructor_standings", "Constructor points going into the race."),
    "cons_pos_before": ("constructor_standings", "Constructor championship position going into the race."),
    "cons_standings_missing": ("constructor_standings", "1 if the constructor has no previous standings row this season."),
    "pts_share": ("driver_standings", "Driver points before the race divided by the leader's points (0 to 1)."),
    "cons_pts_share": ("constructor_standings", "Constructor points before the race divided by the leading constructor's points (0 to 1)."),
    "gap_to_pole_q1_pct": ("qualifying", "Percent slower than the fastest Q1 lap in the same race."),
    "driver_form5": ("form", "Driver's podium rate over the previous 5 races."),
    "constructor_form5": ("form", "Constructor's podium share over the previous 5 races."),
    "circuit_starts_before": ("derived", "Driver's previous starts at this circuit."),
    "circuit_podiums_before": ("derived", "Driver's previous podiums at this circuit."),
    "circuit_podium_rate": ("derived", "Smoothed podium rate at this circuit."),
    "team_circuit_starts_before": ("derived", "Constructor's previous starts at this circuit."),
    "team_circuit_podiums_before": ("derived", "Constructor's previous podiums at this circuit."),
    "home_flag": ("derived", "1 if the driver's nationality matches the circuit country."),
    "era_ordinal":                         ("derived", "Technical era bucket (pre_2006, early_v8, late_v8, hybrid, ground_effect)."),
    "season_progress": ("derived", "Round divided by season length (0 to 1)."),
}

# 2. Prevent pandas from truncating text columns
pd.set_option('display.max_colwidth', None)

# 3. Loop through and display a separate table for each group
for group, cols in FEATURE_GROUPS.items():
    dict_rows = []
    for f in cols:
        # Safeguard in case a feature isn't in your dataframe yet
        if f in model.columns:
            dtype_val = str(model[f].dtype)
            missing_val = round(model[f].isna().mean(), 4)
        else:
            dtype_val = "N/A"
            missing_val = "N/A"
            
        src, note = FEATURE_SOURCE.get(f, ("unknown", "No description provided"))
        dict_rows.append({
            "feature": f,
            "note": note,
            "missing_share": missing_val,
        })
    
    group_df = pd.DataFrame(dict_rows)
    
    print(f"\nFeature Group: {group.upper()}")
    display(group_df)
    print("-" * 60)

In [ ]:
import json, os

# Build the dictionary from FEATURE_GROUPS and FEATURE_SOURCE.
dict_rows = []
for group, cols in FEATURE_GROUPS.items():
    for f in cols:
        src, note = FEATURE_SOURCE[f]
        dict_rows.append({
            "group": group,
            "feature": f,
            "source": src,
            "missing_share": round(model[f].isna().mean(), 4),
            "note": note,
        })
dictionary = pd.DataFrame(dict_rows)

# Save both artifacts.
os.makedirs("../reports", exist_ok=True)

with open("../reports/p3_feature_groups.json", "w") as f:
    json.dump(FEATURE_GROUPS, f, indent=2)

dictionary.to_csv("../reports/p3_data_dictionary.csv", index=False)

# Training settings the model notebooks need (rows before 2006 get a lower weight).
TRAINING_CONFIG = {
    "start_year": START_YEAR,
    "full_weight_from": FULL_WEIGHT_FROM,
    "old_season_weight": OLD_SEASON_WEIGHT,
    "weight_column": "sample_weight",
    "how_to_use": "pipe.fit(X_train, y_train, <final step name>__sample_weight=train['sample_weight'])",
}
with open("../reports/p3_training_config.json", "w") as f:
    json.dump(TRAINING_CONFIG, f, indent=2)

print("saved:")
print(" - reports/p3_feature_groups.json")
print(" - reports/p3_data_dictionary.csv")
print(" - reports/p3_training_config.json")
print(f" - {len(FEATURES)} features across {len(FEATURE_GROUPS)} groups")

In [ ]:
model.to_csv("../reports/p3_modeling_table.csv", index=False)
print("saved modeling table with features, shape:", model.shape)

### Section 3.9: Evidence plots

We produce five plots to justify our feature choices. All of them use the full-weight training seasons (2006–2019), where every qualifying column exists:

1. **Podium rate vs grid bucket** shows that starting position predicts podium.
2. **Podium rate vs qualifying bucket** shows that qualifying position predicts podium.
3. **Feature distributions by target**, in two versions: (a) three candidate features (driver points, team points, circuit podium rate) and (b) three final features (driver and team championship position, gap to pole), showing how each separates podium from non-podium.
4. **Correlation heatmap** of all 20 candidate features checks for redundancy and confirms features relate to `podium`.
5. **Correlation with podium (bar chart)** ranks each feature by its correlation with `podium`, which is easier to read than the full heatmap.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Full-weight training seasons (2006-2019): every qualifying column exists there.
train = model[(model["split"] == "train") & (model["year"] >= FULL_WEIGHT_FROM)].copy()

bins = [0, 2, 4, 6, 10, 15, PIT_GRID - 1, PIT_GRID]
labels = ["1-2", "3-4", "5-6", "7-10", "11-15", "16+", "pit lane"]
train["grid_bucket"] = pd.cut(train["grid_fixed"], bins=bins, labels=labels, right=True)

stats = train.groupby("grid_bucket", observed=True)["podium"].agg(["mean", "count", "sum"])
stats["mean_pct"] = stats["mean"] * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(stats.index.astype(str), stats["mean_pct"], color="tab:blue")
ax.bar_label(bars, labels=[f"{p:.1f}%\n(n={n})" for p, n in
                           zip(stats["mean_pct"], stats["count"])], fontsize=8)
ax.set_title("Podium rate by grid position (training years 2006–2019)")
ax.set_xlabel("Grid position")
ax.set_ylabel("Podium rate (%)")
ax.grid(alpha=0.3, axis="y")
ax.set_ylim(0, stats["mean_pct"].max() * 1.25)
fig.text(0.5, -0.02,
         "Marginal view, not controlled for qualifying or car strength. "
         f"Pit-lane starters carry the placeholder grid value {PIT_GRID}. "
         f"Pit-lane bar is based on only {int(stats.loc['pit lane', 'count'])} starts.",
         ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("../reports/fig_p3_podium_vs_grid.png", dpi=120, bbox_inches="tight")
plt.show()

print(stats.round(3))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Full-weight training seasons (2006-2019): every qualifying column exists there.
train = model[(model["split"] == "train") & (model["year"] >= FULL_WEIGHT_FROM)].copy()

bins = [0, 2, 4, 6, 10, 15, 25]
labels = ["1-2", "3-4", "5-6", "7-10", "11-15", "16+"]
train["qual_bucket"] = pd.cut(train["qual_position"], bins=bins, labels=labels, right=True)

n_missing = train["qual_position"].isna().sum()
stats_q = train.groupby("qual_bucket", observed=True)["podium"].agg(["mean", "count", "sum"])
stats_q["mean_pct"] = stats_q["mean"] * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(stats_q.index.astype(str), stats_q["mean_pct"], color="tab:orange")
ax.bar_label(bars, labels=[f"{p:.1f}%\n(n={n})" for p, n in
                           zip(stats_q["mean_pct"], stats_q["count"])], fontsize=8)
ax.set_title("Podium rate by qualifying position (training years 2006–2019)")
ax.set_xlabel("Qualifying position")
ax.set_ylabel("Podium rate (%)")
ax.grid(alpha=0.3, axis="y")
ax.set_ylim(0, stats_q["mean_pct"].max() * 1.25)
fig.text(0.5, -0.02,
         f"Marginal view, not controlled for grid. {n_missing} training rows with no qualifying position are excluded.",
         ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("../reports/fig_p3_podium_vs_qual.png", dpi=120, bbox_inches="tight")
plt.show()

print(stats_q.round(3))

In [ ]:
# Same plot for both periods: grid matters in every era, but less before 2006.
train_all = model[model["split"] == "train"].copy()
train_all["grid_bucket"] = pd.cut(train_all["grid_fixed"], bins=bins, labels=labels, right=True)
train_all["period"] = np.where(train_all["year"] >= FULL_WEIGHT_FROM, "2006-2019 (weight 1)",
                               f"1950-2005 (weight {OLD_SEASON_WEIGHT})")

rates = (train_all.groupby(["grid_bucket", "period"], observed=True)["podium"]
                  .mean().unstack("period") * 100)

fig, ax = plt.subplots(figsize=(10, 5))
rates.plot(kind="bar", ax=ax, color=["tab:gray", "tab:blue"], width=0.8, rot=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", fontsize=7)
ax.set_title("Podium rate by grid position: old vs modern seasons (training rows)")
ax.set_xlabel("Grid position")
ax.set_ylabel("Podium rate (%)")
ax.grid(alpha=0.3, axis="y")
ax.legend(title="")
plt.tight_layout()
plt.savefig("../reports/fig_p3_podium_vs_grid_by_period.png", dpi=120, bbox_inches="tight")
plt.show()

print(rates.round(1))

In [ ]:
# Plot 3a: three candidate features (points and circuit history).
import numpy as np
import matplotlib.pyplot as plt

# Full-weight training seasons (2006-2019).
train = model[(model["split"] == "train") & (model["year"] >= FULL_WEIGHT_FROM)]

features = ["pts_before", "cons_pts_before", "circuit_podium_rate"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, feat in zip(axes, features):
    x_no = train.loc[train["podium"] == 0, feat].dropna()
    x_yes = train.loc[train["podium"] == 1, feat].dropna()

    # Shared bin edges so the two groups are directly comparable
    edges = np.histogram_bin_edges(pd.concat([x_no, x_yes]), bins=20)

    ax.hist(x_no, bins=edges, density=True, histtype="stepfilled",
            alpha=0.35, color="tab:gray", label=f"not podium (n={len(x_no):,})")
    ax.hist(x_yes, bins=edges, density=True, histtype="step",
            linewidth=3, color="tab:red", label=f"podium (n={len(x_yes):,})")

    ax.set_title(feat)
    ax.set_xlabel(feat)
    ax.set_ylabel("Density (within group)")
    ax.legend(fontsize=8)

fig.suptitle("Candidate features: distributions by podium outcome, training years 2006–2019 "
             "(densities normalised within each group; podium rate ≈ 14%)",
             fontsize=10)
plt.tight_layout()
plt.savefig("../reports/fig_p3_distributions_candidates.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 3b: three final features (championship positions and gap to pole).
import numpy as np
import matplotlib.pyplot as plt

# Full-weight training seasons (2006-2019).
train = model[(model["split"] == "train") & (model["year"] >= FULL_WEIGHT_FROM)]

# Three final features that mean the same in every era (positions and a percentage).
features = [
    ("pos_before", "Driver championship position before the race"),
    ("cons_pos_before", "Team championship position before the race"),
    ("gap_to_pole_pct", "Gap to pole (%)"),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (feat, title) in zip(axes, features):
    x_no = train.loc[train["podium"] == 0, feat].dropna()
    x_yes = train.loc[train["podium"] == 1, feat].dropna()

    if feat == "gap_to_pole_pct":
        # Cut the long tail so the bulk of the data is visible.
        hi = np.percentile(pd.concat([x_no, x_yes]), 99)
        x_no, x_yes = x_no.clip(upper=hi), x_yes.clip(upper=hi)
        edges = np.histogram_bin_edges(pd.concat([x_no, x_yes]), bins=20)
    else:
        # Whole-number bins, so each bar is one championship position.
        edges = np.arange(0.5, pd.concat([x_no, x_yes]).max() + 1.5, 1)

    ax.hist(x_no, bins=edges, density=True, histtype="stepfilled",
            alpha=0.35, color="tab:gray", label=f"not podium (n={len(x_no):,})")
    ax.hist(x_yes, bins=edges, density=True, histtype="step",
            linewidth=3, color="tab:red", label=f"podium (n={len(x_yes):,})")

    ax.set_title(title)
    ax.set_xlabel(feat)
    ax.set_ylabel("Density (within group)")
    ax.legend(fontsize=8)

fig.suptitle("Final features: distributions by podium outcome, full-weight training years 2006–2019 "
             "(densities normalised within each group)", fontsize=10)
plt.tight_layout()
plt.savefig("../reports/fig_p3_distributions_by_target.png", dpi=120, bbox_inches="tight")
plt.show()

# Share of each group in the front of each distribution.
for feat, cut in [("pos_before", 6), ("cons_pos_before", 3), ("gap_to_pole_pct", 1.0)]:
    yes = train.loc[train["podium"] == 1, feat].dropna()
    no = train.loc[train["podium"] == 0, feat].dropna()
    print(f"{feat} <= {cut}: podium {(yes <= cut).mean():.1%}, not podium {(no <= cut).mean():.1%}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Correlations on full-weight TRAINING rows only (2006-2019), for all 20 candidates,
# so validation and test never influence the feature choice.
train = model[(model["split"] == "train") & (model["year"] >= FULL_WEIGHT_FROM)]
print("Features in heatmap:", len(CANDIDATES))
corr = train[CANDIDATES + ["podium"]].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, cbar_kws={"shrink": 0.8}, annot_kws={"size": 7}, ax=ax)
ax.set_title("Feature correlation heatmap (training years 2006–2019 only)")
plt.tight_layout()
plt.savefig("../reports/fig_p3_correlation_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nCorrelation with podium (training rows, sorted):")
print(corr["podium"].drop("podium").sort_values(ascending=False).round(3))

In [ ]:
# Correlation of each feature with podium only (easier to read than the full heatmap)
corr_podium = corr["podium"].drop("podium").sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
colors = ["tab:blue" if v < 0 else "tab:red" for v in corr_podium]
bars = ax.barh(corr_podium.index, corr_podium.values, color=colors)
ax.bar_label(bars, fmt="%+.2f", padding=3, fontsize=8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-0.65, 0.65)
ax.set_title("Correlation of each feature with podium (training years 2006–2019)")
ax.set_xlabel("Correlation with podium (− = lower value, more podiums)")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("../reports/fig_p3_corr_with_podium.png", dpi=120, bbox_inches="tight")
plt.show()

### 3.9 findings

All five plots use the full-weight training rows only (2006–2019). The correlations are computed on the same split, so they are not influenced by validation or test.

**Plot 1 (grid).** Podium rate falls with each step down the grid:

| Grid bucket | Podium rate | Count |
|---|---:|---:|
| 1-2 | 72.9% | 535 |
| 3-4 | 43.7% | 535 |
| 5-6 | 17.5% | 536 |
| 7-10 | 4.9% | 1,068 |
| 11-15 | 1.4% | 1,331 |
| 16+ | 0.9% | 1,717 |
| pit lane (35) | 0.0% | 20 |

The pit-lane group shows no podium finishes, but 20 starts are too few to estimate a rate. Pit-lane starts get `grid_fixed = 35` (`PIT_GRID`), behind every real grid slot; Section 3.11 shows that `pit_lane_flag` adds nothing on top of that.

In the seasons before 2006 the grid matters less (front row 54.1% vs 72.9%, P7–10 11.3% vs 4.9%): races were less predictable, with more retirements. This is one reason the old seasons get a lower training weight.

**Plot 2 (qualifying).** The same shape:

| Qualifying bucket | Podium rate | Count |
|---|---:|---:|
| 1-2 | 71.8% | 536 |
| 3-4 | 43.2% | 535 |
| 5-6 | 20.0% | 535 |
| 7-10 | 4.3% | 1,067 |
| 11-15 | 1.6% | 1,331 |
| 16+ | 0.8% | 1,718 |

Grid and qualifying position are closely related because most drivers start where they qualify. They are not identical: pit-lane starts have a qualifying position but no grid slot, and grid penalties move some drivers back from their qualifying position. 

**Plot 3a (candidate features).**

- `pts_before`: podium drivers have more pre-race points. The podium
  group has a long right tail; the non-podium group is concentrated near
  zero. Clear separation.
- `cons_pts_before`: the same pattern, slightly stronger. The podium
  distribution is shifted further right than the driver version,
  consistent with car performance being the dominant factor.
- `circuit_podium_rate`: the podium group is shifted slightly right, but
  the overlap is large. The feature has signal but is weaker than the
  standings (the circuit group is dropped in Section 3.11).

**Plot 3b (final features).**

- `pos_before`: 87.1% of podium finishers were in the top 6 of the
  championship before the race, against 18.6% of the other drivers.
- `cons_pos_before`: 84.0% of podium finishers drove for a top-3 team,
  against 18.9% of the others, consistent with car performance being the
  dominant factor.
- `gap_to_pole_pct`: 83.9% of podium finishers qualified within 1% of the
  fastest lap, against 19.6% of the others.

**Plot 4 (correlations with podium, training rows only).**

The ranking matches the expected structure. Grid and qualifying are the
strongest features (|r| ≈ 0.50). Driver and constructor standings follow
(|r| ≈ 0.44–0.47). Circuit features are moderate (0.10–0.29). The flags
and `season_progress` are near zero.

- `circuit_podium_rate` (0.285) is stronger than the raw counts
  (`circuit_podiums_before` 0.250, `circuit_starts_before` 0.168),
  supporting the smoothed rate over the counts.
- `best_qual_s` (-0.057) is weak because absolute lap times vary by
  track; `gap_to_pole_pct` (-0.290) is the normalised version and carries
  five times the signal.
- `season_progress` has zero correlation with podium. It was meant to
  help interpret `pts_before` (50 points at round 3 means more than at
  round 20), but logistic regression cannot combine two features on its
  own, so this effect is not captured. It is tested on its own in the
  ablation (Section 3.11).
  `era_ordinal` is 0.012, essentially zero. Era is a time bucket, not a
per-race property, so a linear correlation cannot capture it. It is
tested in the ablation, with the limitation noted in Section 3.7
(validation contains only the hybrid era).
- The rare-event flags (`no_qual_time`, `pit_lane_flag`,
  `*_missing`) have small correlations because they fire on few rows.
  Correlation is the wrong measure for them; the ablation is the right
  one.

No feature pair exceeds 0.95 correlation, so no two features are
redundant enough to drop on correlation alone. `grid_fixed` and
`qual_position` are the highest pair at ~0.9, and both stay: they carry
distinct information, and the ablation (Section 3.11) shows that removing
the grid group lowers validation PR-AUC.

**Plot 5 (correlation with podium, bar chart).** The same correlations as
the `podium` column of the heatmap, sorted, so the ranking is easier to read.

### Section 3.10: Preprocessing experiments

We run four experiments, each with the same logistic regression as the
baseline. Only the preprocessing step changes.

- **1a: scaling vs no scaling.** Does `StandardScaler` change performance
  or convergence?
- **1b: order (impute→scale vs scale→impute).** Does the order matter?
- **2: median vs mean imputation.** Which fill value is better for this
  feature set?
- **3: fit-on-train vs fit-on-all (leakage demo).** Does fitting the
  preprocessor on validation rows inflate the score?

All experiments use the final feature list, the same train/validation split,
the training weights (`sample_weight`, pre-2006 rows 0.25) and the same
`LogisticRegression(max_iter=2000, random_state=SEED)`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

# Ensure splits are defined 
train_df = model[model["split"] == "train"]
val_df   = model[model["split"] == "validation"]
test_df  = model[model["split"] == "test"]

X_tr = train_df[FEATURES];  y_tr = train_df["podium"].values
w_tr = train_df["sample_weight"].values   # pre-2006 rows count less
X_va = val_df[FEATURES];    y_va = val_df["podium"].values

print("X_tr:", X_tr.shape, " X_va:", X_va.shape)
print("\n")

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

def lr_pipeline(scale: bool):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("lr", LogisticRegression(max_iter=2000, random_state=SEED)))
    return Pipeline(steps)

def run_and_count_warnings(pipe, X_fit, y_fit, X_eval, y_eval, w_fit=None):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        pipe.fit(X_fit, y_fit, lr__sample_weight=w_fit)
        proba = pipe.predict_proba(X_eval)[:, 1]
    n_warn = sum(1 for x in w if issubclass(x.category, ConvergenceWarning))
    return {
        "PR-AUC": round(average_precision_score(y_eval, proba), 4),
        "ROC-AUC": round(roc_auc_score(y_eval, proba), 4),
        "warnings": n_warn,
    }

import pandas as pd
rows = []
for name, scale, iters in [("impute→scale", True, 2000),
                           ("impute-only", False, 2000),
                           ("impute-only (max_iter=10000)", False, 10000)]:
    pipe = lr_pipeline(scale=scale)
    pipe.set_params(lr__max_iter=iters)
    r = run_and_count_warnings(pipe, X_tr, y_tr, X_va, y_va, w_fit=w_tr)
    rows.append({"pipeline": name, **r, "steps used": pipe.named_steps["lr"].n_iter_[0]})

exp1a = pd.DataFrame(rows)
print("Experiment 1a: scaling vs no scaling (validation)")
print(exp1a.to_string(index=False))

In [ ]:
pipe_imp_sc = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler()),
    ("lr",  LogisticRegression(max_iter=2000, random_state=SEED)),
])
pipe_sc_imp = Pipeline([
    ("sc",  StandardScaler()),
    ("imp", SimpleImputer(strategy="median")),
    ("lr",  LogisticRegression(max_iter=2000, random_state=SEED)),
])

print("Experiment 1b: order of imputation and scaling")
for name, pipe in [("impute→scale", pipe_imp_sc), ("scale→impute", pipe_sc_imp)]:
    pipe.fit(X_tr, y_tr, lr__sample_weight=w_tr)
    pr = average_precision_score(y_va, pipe.predict_proba(X_va)[:, 1])
    print(f"  {name}: validation PR-AUC = {pr:.4f}")

In [ ]:
def lr_pipeline_imputer(strategy: str):
    return Pipeline([
        ("imputer", SimpleImputer(strategy=strategy)),
        ("scaler",  StandardScaler()),
        ("lr",      LogisticRegression(max_iter=2000, random_state=SEED)),
    ])

rows = []
for name, strategy in [("median", "median"), ("mean", "mean")]:
    pipe = lr_pipeline_imputer(strategy)
    pipe.fit(X_tr, y_tr, lr__sample_weight=w_tr)
    proba = pipe.predict_proba(X_va)[:, 1]
    rows.append({
        "imputer": name,
        "PR-AUC":  round(average_precision_score(y_va, proba), 4),
        "ROC-AUC": round(roc_auc_score(y_va, proba), 4),
    })

exp1c = pd.DataFrame(rows)
print("Experiment 2: median vs mean imputation (validation)")
print(exp1c.to_string(index=False))

In [ ]:
median_imp = SimpleImputer(strategy="median").fit(X_tr)
mean_imp   = SimpleImputer(strategy="mean").fit(X_tr)

compare = pd.DataFrame({
    "feature":       FEATURES,
    "median_fill":   median_imp.statistics_.round(3),
    "mean_fill":     mean_imp.statistics_.round(3),
    "diff":          (mean_imp.statistics_ - median_imp.statistics_).round(3),
    "missing_share": [round(X_tr[f].isna().mean(), 4) for f in FEATURES],
})
print(compare[compare["missing_share"] > 0].to_string(index=False))

In [ ]:
# DEMO ONLY: the "leaky" version below shows what NOT to do.
# The real pipeline (used everywhere else) is fitted on train only.
from sklearn.base import clone

prep_template = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

def lr_on(prep, X, y):
    return LogisticRegression(max_iter=2000, random_state=SEED).fit(
        prep.transform(X), y, sample_weight=w_tr)

prep_correct = clone(prep_template).fit(X_tr)
clf_correct  = lr_on(prep_correct, X_tr, y_tr)
proba_correct = clf_correct.predict_proba(prep_correct.transform(X_va))[:, 1]

X_tr_va = pd.concat([X_tr, X_va])
prep_leaky = clone(prep_template).fit(X_tr_va)
clf_leaky  = lr_on(prep_leaky, X_tr, y_tr)
proba_leaky = clf_leaky.predict_proba(prep_leaky.transform(X_va))[:, 1]

pr_correct = average_precision_score(y_va, proba_correct)
pr_leaky   = average_precision_score(y_va, proba_leaky)

print("Experiment 3: preprocessing fit on train vs train+val")
print(f"  Correct (train only) : PR-AUC = {pr_correct:.4f}")
print(f"  Leaky   (train+val)  : PR-AUC = {pr_leaky:.4f}")
print(f"  Difference           : {pr_leaky - pr_correct:+.4f}")

scaler_correct = prep_correct.named_steps["scaler"]
scaler_leaky   = prep_leaky.named_steps["scaler"]
mean_shift_std  = np.abs(scaler_correct.mean_ - scaler_leaky.mean_) / scaler_correct.scale_
scale_shift_pct = np.abs(scaler_correct.scale_ - scaler_leaky.scale_) / scaler_correct.scale_ * 100

print(f"\nScaler drift: mean shift {mean_shift_std.mean():.4f} std (max {mean_shift_std.max():.4f})")
print(f"              scale change {scale_shift_pct.mean():.2f}% (max {scale_shift_pct.max():.2f}%)")

### 3.10 findings

All experiments use the final 12 features, training rows 1950–2019 with the
season weights, and validation 2020–2021.

**Experiment 1a: scaling vs no scaling.**

| Pipeline | PR-AUC | ROC-AUC | Convergence warnings | Iterations used |
|---|---:|---:|---:|---:|
| impute→scale | 0.7351 | 0.9273 | 0 | 30 |
| impute-only | 0.7351 | 0.9272 | 0 | 696 |
| impute-only (max_iter=10000) | 0.7351 | 0.9272 | 0 | 696 |

Both pipelines converge and score the same. We keep `impute→scale`
because it converges about 23 times faster (30 vs 696 iterations), makes
the L2 penalty treat all features equally (otherwise a feature in the
hundreds, like `pts_before`, is penalised differently from a 0/1 flag), and
is required by the other models we train (e.g. neural networks).

**Experiment 1b: order of imputation and scaling.**

| Order | Validation PR-AUC |
|---|---:|
| impute→scale | 0.7351 |
| scale→impute | 0.7349 |

The same to 3 decimals. We use `impute→scale`, the standard scikit-learn
order, which produces a final distribution with mean 0 and std 1.

**Experiment 2: median vs mean imputation (validation).**

| Feature | Median fill | Mean fill | Difference | Missing in train |
|---|---:|---:|---:|---:|
| `qual_position` | 11.000 | 11.345 | +0.345 | 63.1% |
| `gap_to_pole_pct` | 1.961 | 2.503 | +0.542 | 63.7% |
| `pos_before` | 12.000 | 13.072 | +1.072 | 12.4% |
| `cons_pos_before` | 6.000 | 6.162 | +0.162 | 12.4% |

Validation PR-AUC is identical for both (0.7351 / 0.7351). The missing
shares are high because of the old seasons (no qualifying data before 1994,
no constructors' championship before 1958). Those rows are marked by
`no_qual_time` and the standings flags and have weight 0.25, while the
validation rows (2020–21) are almost never missing, so the fill value hardly
moves the score. We choose **median**, the standard choice for skewed
features (`gap_to_pole_pct` is right-skewed). Note that `SimpleImputer` is
not weighted: the medians are computed over all training rows.

**Experiment 3: fit on train vs train+val (leakage demo).**

| Setup | Validation PR-AUC |
|---|---:|
| Fit on train only (correct) | 0.7351 |
| Fit on train+val (leaky) | 0.7350 |
| Difference | −0.0001 |

The metric impact is small, but the scaler's parameters drift measurably:
per-feature mean shift 0.0188 std on average (max 0.0433), scale change
2.24% on average (max 5.45%). The leak is real. We fit on train only because
it is the honest rule and the effect could be larger under a bigger
distribution shift.

### Section 3.11: Feature ablation

We measure what each feature and each feature group adds by changing the
feature set and retraining the same model. Everything else stays fixed: the
same pipeline (median imputation, scaling, logistic regression), the same
training data (1950 onwards, pre-2006 rows weighted 0.25) and the same
metric (validation PR-AUC from `evaluate()` in `src/f1_eval.py`). The model
is fitted here with the weights, then scored with `evaluate()`, because
`train_eval()` does not take weights.

**Three validation periods.** A single validation period can move by about
0.001 by chance, so every test is run on three expanding time folds. The test
years (2022+) are never used.

| Fold | Train | Validate |
|---|---|---|
| 1 | 1950–2015 | 2016–2017 |
| 2 | 1950–2017 | 2018–2019 |
| 3 | 1950–2019 | 2020–2021 (the official validation split) |

**Feature lists.** `CANDIDATE_GROUPS` (20 candidates) and `FEATURE_GROUPS`
(final list) are defined in Section 3.8.

- **Part A: group ablation.** Remove one candidate group at a time, plus grid and
  qualifying together (they are about 0.9 correlated, so each can cover for the other).
- **Part B: single-feature tests**, starting from the 20 candidates: *remove*
  (is the feature needed?), *swap* (is a new version better?) and *add*
  (does a new candidate help?).
- **Part D: three focused questions**: grid encoding, qualifying rank vs round
  reached, season points vs last-5-races form.
- **Part F: using the seasons before 2006**: dropped, full weight, or down-weighted.
- **Part C: the final list** against the 20 candidates, and a group ablation on it.
- **Part E: extra checks** on the final list: add each unused column, three new feature
  ideas (teammate gap, reliability, places gained), every subset of the final list, and a
  grid fallback for the missing qualifying positions of the old seasons.

**Decision rule.** A change counts only if it moves mean PR-AUC by more than
`MIN_GAIN = 0.002` and goes the same way in at least 2 of the 3 folds.
Otherwise the simpler feature set wins.

**Limitation for `era_ordinal`.** No fold contains the ground-effect era, so
the ablation can only show its effect on earlier eras, not whether the test
value (4) is safe.

In [ ]:
import sys
sys.path.append("../src")
from f1_eval import evaluate

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# CANDIDATE_GROUPS / CANDIDATES (20) and FEATURE_GROUPS / FEATURES (final) come from Section 3.8.
abl = model.copy()

# Three expanding time folds. Training starts in START_YEAR; test years (2022+) are never used.
FOLDS = [
    ("2016-17", abl["year"] <= 2015, abl["year"].between(2016, 2017)),
    ("2018-19", abl["year"] <= 2017, abl["year"].between(2018, 2019)),
    ("2020-21", abl["year"] <= 2019, abl["year"].between(2020, 2021)),
]
FOLD_NAMES = [name for name, _, _ in FOLDS]
MIN_GAIN = 0.002

def make_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000, random_state=SEED)),
    ])

def cv_pr_auc(features, start=START_YEAR, old_weight=OLD_SEASON_WEIGHT):
    """Validation PR-AUC on each of the three folds.
    Training rows from `start`; rows before FULL_WEIGHT_FROM get `old_weight`."""
    scores = []
    for name, tr, va in FOLDS:
        tr = tr & (abl["year"] >= start)
        w = np.where(abl.loc[tr, "year"] < FULL_WEIGHT_FROM, old_weight, 1.0)
        pipe = make_pipeline()
        pipe.fit(abl.loc[tr, features], abl.loc[tr, "podium"].values, lr__sample_weight=w)
        scores.append(evaluate(pipe, abl.loc[va, features], abl.loc[va, "podium"].values)["PR-AUC"])
    return np.array(scores)

base_scores = cv_pr_auc(CANDIDATES)
print(f"Training from {START_YEAR}, pre-{FULL_WEIGHT_FROM} weight {OLD_SEASON_WEIGHT}")
print(f"Candidates: {len(CANDIDATES)} features in {len(CANDIDATE_GROUPS)} groups")
for name, s in zip(FOLD_NAMES, base_scores):
    print(f"  baseline PR-AUC, validate {name}: {s:.4f}")
print(f"  mean: {base_scores.mean():.4f}")

In [ ]:
# Part A: remove one group at a time (plus grid and qualifying together).
# drop = baseline - score without the group, so a positive drop means the group helps.
group_tests = dict(CANDIDATE_GROUPS)
group_tests["grid + qualifying"] = CANDIDATE_GROUPS["grid"] + CANDIDATE_GROUPS["qualifying"]

rows = []
for group, cols in group_tests.items():
    drop = base_scores - cv_pr_auc([f for f in CANDIDATES if f not in cols])
    rows.append({
        "dropped group": group,
        **{f"drop {n}": d for n, d in zip(FOLD_NAMES, drop)},
        "mean drop": drop.mean(),
        "folds where it helps": int((drop > 0).sum()),
    })

group_ablation = pd.DataFrame(rows).sort_values("mean drop", ascending=False)
print(group_ablation.round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

# Mean drop per group, with each fold as a dot so the spread is visible.
plot_df = group_ablation.sort_values("mean drop")
fold_cols = [f"drop {n}" for n in FOLD_NAMES]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_df["dropped group"], plot_df["mean drop"], color="tab:blue", alpha=0.8,
        label="mean over 3 folds")
for k, col in enumerate(fold_cols):
    ax.scatter(plot_df[col], plot_df["dropped group"], color="black", s=14,
               marker="o", zorder=3, label="single fold" if k == 0 else None)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xscale("symlog", linthresh=0.005)
ax.set_title("Group ablation: PR-AUC drop when the group is removed (3 validation folds)")
ax.set_xlabel("PR-AUC drop vs baseline (larger = more important; symlog scale)")
ax.grid(alpha=0.3, axis="x")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig("../reports/fig_p3_ablation.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Part B: single-feature tests, each starting from the 20 candidates.
# gain = score after the change - baseline, so a positive gain means the change helps.
def without(*cols):
    return [f for f in CANDIDATES if f not in cols]

def swap(old, new):
    return [new if f == old else f for f in CANDIDATES]

TESTS = [
    ("remove", "best_qual_s",                  without("best_qual_s")),
    ("remove", "raw circuit counts (2)",       without("circuit_starts_before", "circuit_podiums_before")),
    ("remove", "circuit_podium_rate",          without("circuit_podium_rate")),
    ("remove", "team circuit history (2)",     without("team_circuit_starts_before", "team_circuit_podiums_before")),
    ("remove", "home_flag",                    without("home_flag")),
    ("remove", "season_progress",              without("season_progress")),
    ("remove", "era_ordinal",                  without("era_ordinal")),
    ("swap",   "pts_before -> pts_share",      swap("pts_before", "pts_share")),
    ("swap",   "cons_pts_before -> cons_pts_share", swap("cons_pts_before", "cons_pts_share")),
    ("swap",   "gap_to_pole_pct -> gap_to_pole_q1_pct", swap("gap_to_pole_pct", "gap_to_pole_q1_pct")),
    ("add",    "qual_group (2 yes/no columns)", CANDIDATES + QUAL_GROUP_COLS),
    ("add",    "driver_form5",                 CANDIDATES + ["driver_form5"]),
    ("add",    "constructor_form5",            CANDIDATES + ["constructor_form5"]),
    ("add",    "both form features",           CANDIDATES + ["driver_form5", "constructor_form5"]),
]

def decide(kind, gain):
    clear_gain = gain.mean() > MIN_GAIN and (gain > 0).sum() >= 2
    clear_loss = gain.mean() < -MIN_GAIN and (gain < 0).sum() >= 2
    if kind == "remove":
        return "keep" if clear_loss else "drop"
    if kind == "swap":
        return "use new" if clear_gain else "keep current"
    return "add" if clear_gain else "do not add"

rows = []
for kind, name, features in TESTS:
    gain = cv_pr_auc(features) - base_scores
    rows.append({
        "test": kind,
        "change": name,
        **{f"gain {n}": g for n, g in zip(FOLD_NAMES, gain)},
        "mean gain": gain.mean(),
        "decision": decide(kind, gain),
    })

feature_tests = pd.DataFrame(rows)
print(feature_tests.round(4).to_string(index=False))

In [ ]:
# Part D: focused comparisons, each starting from the 20 candidates.
# Grid encoding: is the fix (pit lane = 25), the flag, or both doing the work?
# Rank vs round reached: qual_position, qual_group, or both?
# Points vs recent form: season points, last-5-races form, or both?
# (pos_before / cons_pos_before stay in every variant; only the points columns change.)
def replace(features, old, new):
    out = [f for f in features if f not in old]
    return out + [f for f in new if f not in out]

COMPARISONS = {
    "Grid encoding": [
        ("grid_fixed + pit_lane_flag (current)", CANDIDATES),
        ("grid_fixed only",                      without("pit_lane_flag")),
        ("raw grid (0 = pit lane) + pit_lane_flag", replace(CANDIDATES, ["grid_fixed"], ["grid"])),
        ("raw grid only (original version)",     replace(CANDIDATES, ["grid_fixed", "pit_lane_flag"], ["grid"])),
    ],
    "Qualifying rank vs round reached": [
        ("qual_position only (current)",         CANDIDATES),
        ("qual_group only",                      replace(CANDIDATES, ["qual_position"], QUAL_GROUP_COLS)),
        ("qual_position + qual_group",           CANDIDATES + QUAL_GROUP_COLS),
    ],
    "Points vs recent form": [
        ("season points (current)",              CANDIDATES),
        ("driver: form instead of points",       replace(CANDIDATES, ["pts_before"], ["driver_form5"])),
        ("team: form instead of points",         replace(CANDIDATES, ["cons_pts_before"], ["constructor_form5"])),
        ("both: form instead of points",         replace(CANDIDATES, ["pts_before", "cons_pts_before"],
                                                         ["driver_form5", "constructor_form5"])),
        ("points + form",                        CANDIDATES + ["driver_form5", "constructor_form5"]),
    ],
}

comparison_tables = {}
for topic, variants in COMPARISONS.items():
    rows = []
    for name, features in variants:
        scores = cv_pr_auc(features)
        gain = scores - base_scores
        rows.append({
            "variant": name,
            **{f"PR-AUC {n}": s for n, s in zip(FOLD_NAMES, scores)},
            "mean PR-AUC": scores.mean(),
            "gain vs current": gain.mean(),
            "folds better": int((gain > 0).sum()),
        })
    comparison_tables[topic] = pd.DataFrame(rows)
    print(f"\n{topic}")
    print(comparison_tables[topic].round(4).to_string(index=False))

In [ ]:
# Part F: using the seasons before 2006.
# Same 3 folds and features; only the training years and the weight of pre-2006 rows change.
SEASON_SETUPS = [
    ("2006-2019 only (old seasons dropped)",    FULL_WEIGHT_FROM, 1.0),
    ("1950-2019, no weights",                   1950, 1.0),
    ("1950-2019, pre-2006 weight 0.5",          1950, 0.5),
    ("1950-2019, pre-2006 weight 0.25 (used)",  1950, 0.25),
    ("1950-2019, pre-2006 weight 0.1",          1950, 0.1),
]

season_tables = {}
for set_name, features in [("20 candidates", CANDIDATES), ("Final list", FEATURES)]:
    rows = []
    for name, start, w_old in SEASON_SETUPS:
        scores = cv_pr_auc(features, start=start, old_weight=w_old)
        rows.append({"training data": name,
                     **{f"PR-AUC {n}": s for n, s in zip(FOLD_NAMES, scores)},
                     "mean PR-AUC": scores.mean()})
    table = pd.DataFrame(rows)
    no_w = table.loc[table["training data"] == "1950-2019, no weights", "mean PR-AUC"].iloc[0]
    table["vs 1950 no weights"] = table["mean PR-AUC"] - no_w
    season_tables[set_name] = table
    print(f"\n{set_name} ({len(features)} features)")
    print(table.round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

# Part F chart: mean PR-AUC for each way of using the old seasons (final list).
t = season_tables["Final list"]
fig, ax = plt.subplots(figsize=(9, 3.6))
colors = ["tab:gray", "tab:red", "tab:blue", "tab:blue", "tab:blue"]
bars = ax.barh(t["training data"], t["mean PR-AUC"], color=colors)
ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
ax.invert_yaxis()
ax.set_xlim(t["mean PR-AUC"].min() - 0.01, t["mean PR-AUC"].max() + 0.006)
ax.set_title("Final list: mean validation PR-AUC over 3 periods, by training data")
ax.set_xlabel("Mean PR-AUC (zoomed axis)")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("../reports/fig_p3_seasons_weights.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Part C: the final list (FEATURE_GROUPS, Option 1) against the 20 candidates,
# then a group ablation on the final list.
final_scores = cv_pr_auc(FEATURES)
compare_final = pd.DataFrame({
    "feature set": [f"20 candidates", f"Final list ({len(FEATURES)})"],
    **{f"PR-AUC {n}": [b, f] for n, b, f in zip(FOLD_NAMES, base_scores, final_scores)},
    "mean PR-AUC": [base_scores.mean(), final_scores.mean()],
})
print(compare_final.round(4).to_string(index=False))

rows = []
for group, cols in FEATURE_GROUPS.items():
    drop = final_scores - cv_pr_auc([f for f in FEATURES if f not in cols])
    rows.append({"dropped group": group,
                 **{f"drop {n}": d for n, d in zip(FOLD_NAMES, drop)},
                 "mean drop": drop.mean(), "folds where it helps": int((drop > 0).sum())})
gq = FEATURE_GROUPS["grid"] + FEATURE_GROUPS["qualifying"]
drop = final_scores - cv_pr_auc([f for f in FEATURES if f not in gq])
rows.append({"dropped group": "grid + qualifying",
             **{f"drop {n}": d for n, d in zip(FOLD_NAMES, drop)},
             "mean drop": drop.mean(), "folds where it helps": int((drop > 0).sum())})
final_ablation = pd.DataFrame(rows).sort_values("mean drop", ascending=False)
print("\nGroup ablation on the final list:")
print(final_ablation.round(4).to_string(index=False))

In [ ]:
# Part C (continued): remove each final feature on its own.
# drop = final score - score without the feature, so a positive drop means the feature helps.
# The two qual_group columns are also removed together, since they describe one thing.
single_tests = [(f, [f]) for f in FEATURES] + [("qg_q3 + qg_q2_only (removed together)", QUAL_GROUP_COLS)]

rows = []
for name, cols in single_tests:
    drop = final_scores - cv_pr_auc([f for f in FEATURES if f not in cols])
    group = next((g for g, fs in FEATURE_GROUPS.items() if cols[0] in fs), "")
    rows.append({
        "removed feature": name,
        "group": group,
        **{f"drop {n}": d for n, d in zip(FOLD_NAMES, drop)},
        "mean drop": drop.mean(),
        "helps in": f"{int((drop > 0).sum())} of 3",
        "verdict": ("clearly needed" if drop.mean() > MIN_GAIN and (drop > 0).sum() >= 2
                    else "small or shared"),
    })

feature_ablation = pd.DataFrame(rows).sort_values("mean drop", ascending=False)
print(f"Final list: {len(FEATURES)} features, mean PR-AUC {final_scores.mean():.4f}\n")
print(feature_ablation.round(4).to_string(index=False))

In [ ]:
# Part E1: add each unused column of the table to the final list, one at a time.
# Includes the dropped candidates, the extra candidates, raw columns and IDs
# (IDs are labels, not quantities, so they are only here to show they add nothing).
UNUSED = [c for c in CANDIDATES + ["pts_share", "cons_pts_share", "gap_to_pole_q1_pct",
                                   "driver_form5", "constructor_form5"] if c not in FEATURES]
UNUSED += ["grid", "q1_s", "q2_s", "q3_s", "year", "round", "number",
           "driverId", "constructorId", "circuitId"]

rows = []
for col in UNUSED:
    gain = cv_pr_auc(FEATURES + [col]) - final_scores
    rows.append({
        "added column": col,
        **{f"gain {n}": g for n, g in zip(FOLD_NAMES, gain)},
        "mean gain": gain.mean(),
        "folds better": int((gain > 0).sum()),
        "decision": "add" if gain.mean() > MIN_GAIN and (gain > 0).sum() >= 2 else "do not add",
    })
add_one = pd.DataFrame(rows).sort_values("mean gain", ascending=False)
print(f"Final list: {len(FEATURES)} features, mean PR-AUC {final_scores.mean():.4f}\n")
print(add_one.round(4).to_string(index=False))

In [ ]:
# Part E2: three new feature ideas, built here as candidates only (on the ablation copy).
# - qual_gap_to_teammate: the driver's gap to pole minus the teammate's, same race (pre-race).
# - team_dnf_rate10 / driver_dnf_rate10: share of the previous 10 races not finished.
# - pos_gained_avg10: average places gained (grid - finish) over the previous 10 finished races.
# The last three use race results (status, finishing order), so they are shifted by one race
# before the rolling window: a row only ever sees earlier races.
team = abl.groupby(["raceId", "constructorId"])["gap_to_pole_pct"]
others_sum = team.transform("sum") - abl["gap_to_pole_pct"].fillna(0)
others_n = team.transform("count") - abl["gap_to_pole_pct"].notna().astype(int)
abl["qual_gap_to_teammate"] = abl["gap_to_pole_pct"] - (others_sum / others_n.replace(0, np.nan))

hist = base[["raceId", "driverId", "constructorId", "year", "round", "grid"]].merge(
    outcomes[["raceId", "driverId", "positionOrder", "status"]],
    on=["raceId", "driverId"], how="left", validate="one_to_one")
finished = hist["status"].eq("Finished") | hist["status"].str.match(r"^\+\d+ Laps?$", na=False)
hist["dnf"] = (~finished).astype(float)
hist["places_gained"] = (hist["grid"] - hist["positionOrder"]).where(finished & (hist["grid"] > 0))

team_races = (hist.groupby(["constructorId", "raceId", "year", "round"], as_index=False)["dnf"].mean()
                  .sort_values(["constructorId", "year", "round"]).reset_index(drop=True))
team_races["team_dnf_rate10"] = team_races.groupby("constructorId")["dnf"].transform(
    lambda s: s.shift(1).rolling(10, min_periods=3).mean())

hist = hist.sort_values(["driverId", "year", "round"]).reset_index(drop=True)
hist["driver_dnf_rate10"] = hist.groupby("driverId")["dnf"].transform(
    lambda s: s.shift(1).rolling(10, min_periods=3).mean())
hist["pos_gained_avg10"] = hist.groupby("driverId")["places_gained"].transform(
    lambda s: s.shift(1).rolling(10, min_periods=3).mean())

# Leakage checks: no history on a first race.
assert team_races.loc[team_races.groupby("constructorId").cumcount() == 0, "team_dnf_rate10"].isna().all()
first_d = hist.groupby("driverId").cumcount() == 0
assert hist.loc[first_d, ["driver_dnf_rate10", "pos_gained_avg10"]].isna().all().all()

abl = abl.drop(columns=["team_dnf_rate10", "driver_dnf_rate10", "pos_gained_avg10"], errors="ignore")
abl = (abl.merge(team_races[["raceId", "constructorId", "team_dnf_rate10"]],
                 on=["raceId", "constructorId"], how="left", validate="many_to_one")
          .merge(hist[["raceId", "driverId", "driver_dnf_rate10", "pos_gained_avg10"]],
                 on=["raceId", "driverId"], how="left", validate="one_to_one"))

NEW_IDEAS = {
    "qual_gap_to_teammate": ["qual_gap_to_teammate"],
    "team_dnf_rate10": ["team_dnf_rate10"],
    "driver_dnf_rate10": ["driver_dnf_rate10"],
    "pos_gained_avg10": ["pos_gained_avg10"],
    "all four": ["qual_gap_to_teammate", "team_dnf_rate10", "driver_dnf_rate10", "pos_gained_avg10"],
}
rows = []
for name, cols in NEW_IDEAS.items():
    gain = cv_pr_auc(FEATURES + cols) - final_scores
    rows.append({
        "added": name,
        **{f"gain {n}": g for n, g in zip(FOLD_NAMES, gain)},
        "mean gain": gain.mean(),
        "folds better": int((gain > 0).sum()),
        "decision": "add" if gain.mean() > MIN_GAIN and (gain > 0).sum() >= 2 else "do not add",
    })
new_ideas = pd.DataFrame(rows)
print(new_ideas.round(4).to_string(index=False))

In [ ]:
# Part E3: every subset of the final list (the two qg columns count as one unit),
# to check that no smaller combination clearly beats it. Takes a few minutes.
import itertools

UNITS = {f: [f] for f in FEATURES if f not in QUAL_GROUP_COLS}
UNITS["qg_q3 + qg_q2_only"] = QUAL_GROUP_COLS
unit_names = list(UNITS)

rows = []
for k in range(1, len(unit_names) + 1):
    for combo in itertools.combinations(unit_names, k):
        scores = cv_pr_auc([c for u in combo for c in UNITS[u]])
        rows.append({
            "dropped": ", ".join(u for u in unit_names if u not in combo) or "(none: final list)",
            **{f"PR-AUC {n}": s for n, s in zip(FOLD_NAMES, scores)},
            "mean PR-AUC": scores.mean(),
            "vs final": scores.mean() - final_scores.mean(),
            "folds better": int((scores > final_scores).sum()),
        })

subsets = pd.DataFrame(rows).sort_values("mean PR-AUC", ascending=False).reset_index(drop=True)
rank = int(subsets.index[subsets["dropped"] == "(none: final list)"][0]) + 1
clear_winners = subsets[(subsets["vs final"] > MIN_GAIN) & (subsets["folds better"] >= 2)]

print(f"Subsets tested: {len(subsets)}  |  rank of the final list by mean: {rank}")
print(f"Subsets beating the final list by more than {MIN_GAIN} in at least 2 folds: {len(clear_winners)}\n")
print("Top 10 by mean PR-AUC:")
print(subsets.head(10).round(4).to_string(index=False))
best_2021 = subsets.sort_values("PR-AUC 2020-21", ascending=False).head(3)
print("\nTop 3 on the official validation period (2020-21):")
print(best_2021.round(4).to_string(index=False))

In [ ]:
# Part E4: grid fallback for the missing qualifying positions of the old seasons.
# Before 2006 most rows have no qualifying position; today they get the median (via the
# imputer) plus no_qual_time = 1. Alternative: use the starting grid slot instead.
abl["qual_position_grid_fallback"] = abl["qual_position"].fillna(
    abl["grid"].where(abl["grid"] > 0)).astype(float)

train_rows = abl["year"] <= 2019
print(f"qual_position missing in training rows: {abl.loc[train_rows, 'qual_position'].isna().mean():.1%} "
      f"-> with grid fallback: {abl.loc[train_rows, 'qual_position_grid_fallback'].isna().mean():.1%}")

fallback_scores = cv_pr_auc([("qual_position_grid_fallback" if f == "qual_position" else f)
                             for f in FEATURES])
grid_fallback = pd.DataFrame({
    "qual_position filled with": ["median + no_qual_time flag (final list)", "grid slot (fallback)"],
    **{f"PR-AUC {n}": [a, b] for n, a, b in zip(FOLD_NAMES, final_scores, fallback_scores)},
    "mean PR-AUC": [final_scores.mean(), fallback_scores.mean()],
})
print(grid_fallback.round(4).to_string(index=False))

### 3.11 findings

All results: 3 validation periods, training from 1950 with pre-2006 rows
weighted 0.25, logistic regression. Drop / gain = change in validation PR-AUC.

**Part A: group ablation** (20 candidates, baseline mean 0.7248).

| Dropped group | Drop 2016-17 | Drop 2018-19 | Drop 2020-21 | Mean drop | Helps in |
|---|---:|---:|---:|---:|---:|
| grid + qualifying | +0.1041 | +0.0934 | +0.0948 | +0.0974 | 3 of 3 |
| grid | +0.0288 | +0.0193 | +0.0137 | +0.0206 | 3 of 3 |
| driver_standings | −0.0136 | +0.0141 | +0.0400 | +0.0135 | 2 of 3 |
| qualifying | +0.0014 | +0.0114 | +0.0058 | +0.0062 | 3 of 3 |
| context | +0.0038 | +0.0005 | +0.0044 | +0.0029 | 3 of 3 |
| circuit | −0.0022 | +0.0030 | +0.0046 | +0.0018 | 2 of 3 |
| constructor_standings | −0.0021 | +0.0013 | +0.0039 | +0.0010 | 2 of 3 |

Grid and qualifying cover for each other: removed one at a time they cost
0.021 and 0.006, removed together 0.097.

**Part B: single-feature tests.**

- *Remove*: `best_qual_s`, the raw circuit counts, `circuit_podium_rate`,
  team circuit history, `home_flag` and `era_ordinal` are not needed
  (|mean change| ≤ 0.0017). Only `season_progress` passes the rule: removing it
  costs 0.0024, in all 3 periods.
- *Swap*: `pts_share` (−0.0030), `cons_pts_share` (−0.0033) and the Q1-only gap
  (−0.0018) are worse than the current versions.
- *Add*: `qual_group` +0.0007 (a tie); the form features hurt (−0.0034 to −0.0058).

**Part D: focused questions.**

- *Grid encoding*: `grid_fixed` alone scores the same as `grid_fixed` + `pit_lane_flag`
  (0.7248); without any pit-lane fix (the original bug) the score drops by 0.0039.
- *Rank vs round reached*: `qual_group` only +0.0004, both +0.0007: a tie.
- *Points vs form*: season points win; form instead of points costs 0.0053–0.0080.

**Part F: using the seasons before 2006** (final list).

| Training data | 2016-17 | 2018-19 | 2020-21 | Mean | vs 1950 no weights |
|---|---:|---:|---:|---:|---:|
| 2006–2019 only (old seasons dropped) | 0.7336 | 0.7250 | 0.7415 | 0.7334 | +0.0131 |
| 1950–2019, no weights | 0.7360 | 0.6971 | 0.7278 | 0.7203 | — |
| 1950–2019, pre-2006 weight 0.5 | 0.7348 | 0.7017 | 0.7309 | 0.7225 | +0.0022 |
| **1950–2019, pre-2006 weight 0.25 (used)** | 0.7358 | 0.7045 | 0.7351 | **0.7251** | **+0.0049** |
| 1950–2019, pre-2006 weight 0.1 | 0.7355 | 0.7055 | 0.7346 | 0.7252 | +0.0049 |

Using every season at full weight is the worst option (0.7203). Down-weighting
the old seasons recovers part of the loss: weight 0.25 is 0.0049 above no
weights, and weight 0.1 gives the same. Dropping the old seasons still
scores highest on validation (+0.0083 above weight 0.25): they have no
qualifying data and come from a different version of the sport, and they help
the oldest validation period (2016–17) but not the two most recent ones. We
keep every season with weight 0.25, so no data is discarded, at a cost of
0.008 mean PR-AUC compared with dropping them. The 20 candidates show the same
pattern (no weights 0.7187, weight 0.25 0.7248, dropped 0.7277).

**Part C: the final list.**

| Feature set | 2016-17 | 2018-19 | 2020-21 | Mean |
|---|---:|---:|---:|---:|
| 20 candidates | 0.7338 | 0.7028 | 0.7377 | 0.7248 |
| Final list (12) | 0.7358 | 0.7045 | 0.7351 | 0.7251 |

The final list ties the 20 candidates with 8 fewer features, so the simpler
list wins. Group ablation on the final list:

| Dropped group | Drop 2016-17 | Drop 2018-19 | Drop 2020-21 | Mean drop | Helps in |
|---|---:|---:|---:|---:|---:|
| grid + qualifying | +0.0950 | +0.0879 | +0.0856 | +0.0895 | 3 of 3 |
| driver_standings | −0.0169 | +0.0165 | +0.0556 | +0.0184 | 2 of 3 |
| grid | +0.0282 | +0.0135 | +0.0090 | +0.0169 | 3 of 3 |
| constructor_standings | −0.0084 | +0.0113 | +0.0140 | +0.0057 | 2 of 3 |
| qualifying | +0.0042 | −0.0049 | +0.0147 | +0.0047 | 2 of 3 |

All four groups contribute (each costs at least 0.0047 when removed, in at
least 2 periods).

**Single features in the final list.** Removing one feature at a time:

| Removed feature | Drop 2016-17 | Drop 2018-19 | Drop 2020-21 | Mean drop | Verdict |
|---|---:|---:|---:|---:|---|
| `grid_fixed` | +0.0282 | +0.0135 | +0.0090 | +0.0169 | clearly needed |
| `pts_before` | −0.0163 | +0.0099 | +0.0338 | +0.0091 | clearly needed |
| `cons_pts_before` | −0.0128 | +0.0114 | +0.0187 | +0.0058 | clearly needed |
| `pos_before` | +0.0085 | +0.0004 | +0.0017 | +0.0035 | clearly needed |
| `cons_standings_missing` | +0.0074 | +0.0011 | +0.0012 | +0.0033 | clearly needed |
| `cons_pos_before` | −0.0067 | +0.0106 | +0.0058 | +0.0032 | clearly needed |
| `qual_position` | −0.0039 | +0.0050 | +0.0072 | +0.0027 | clearly needed |
| `no_qual_time` | −0.0016 | +0.0009 | +0.0049 | +0.0014 | small or shared |
| `qg_q3` | +0.0047 | −0.0016 | +0.0004 | +0.0012 | small or shared |
| `qg_q3` + `qg_q2_only` (removed together) | +0.0033 | −0.0016 | +0.0016 | +0.0011 | small or shared |
| `gap_to_pole_pct` | +0.0029 | −0.0019 | +0.0019 | +0.0010 | small or shared |
| `qg_q2_only` | +0.0009 | +0.0009 | +0.0010 | +0.0009 | small or shared |
| `standings_missing` | +0.0008 | +0.0001 | −0.0000 | +0.0003 | small or shared |

Seven features are clearly needed on their own. The others are small on
their own because they share information (`gap_to_pole_pct` overlaps with
`qual_position` and `grid_fixed`; the missing flags affect few rows), but none
of them hurts, and the qualifying group as a whole costs 0.0047 when removed.

**`qg_q1_only` removed.** `qg_q3 + qg_q2_only + qg_q1_only + no_qual_time = 1`
on all 24,566 rows, so `qg_q1_only` can be derived from the other three.
With all four columns, removing any single one changed nothing (−0.0002);
without `qg_q1_only` the score is the same (0.7251 vs 0.7249 with 13
features) and every remaining qualifying column now shows a positive effect.

Notes on the final list:
- `qual_group` is kept: it clearly helped when training on the knockout seasons
  only (+0.0048 on 2006–2019), and it is a tie with weighted 1950 training
  because the "round reached" only exists in the knockout format.
- `season_progress` is not added: removing it from the 20 candidates costs
  0.0024, but adding it to the final list gains only 0.0005, below the rule.
- `pit_lane_flag` adds nothing (0.0000): `grid_fixed = PIT_GRID` already handles pit-lane starts.

**Part E: extra checks on the final list** (mean 0.7251).

- *E1, add one unused column* (25 columns): none passes the rule. The best
  are `pts_share` (+0.0012), `round` (+0.0009) and `circuit_podiums_before`
  (+0.0008). The IDs hurt (`number` −0.0160), as expected for labels.
- *E2, new feature ideas*: `qual_gap_to_teammate` +0.0000, `team_dnf_rate10`
  −0.0010, `driver_dnf_rate10` −0.0009, `pos_gained_avg10` +0.0002, all four
  +0.0001. None helps: the standings already reflect reliability, and the
  drivers who gain many places are mostly those who start at the back.
- *E3, every subset of the final list* (2,047 subsets, the two qg columns as
  one unit): the final list ranks 41st by mean. Three subsets pass the rule
  against it (+0.0021 to +0.0027 mean, better in 2 of 3 periods). All three
  drop the season points (`pts_before`, `cons_pts_before`) and keep the
  positions. They win on 2016–17 and 2018–19 but lose 0.015–0.017 on 2020–21,
  the official validation period and the closest to the test years. With
  2,047 subsets tested, a few passing the 0.002 rule by chance is expected,
  and `pts_before` has the largest single drop on 2020–21 (+0.0338). We keep
  the final list.
- *E4, grid fallback for the old seasons*: filling the missing
  `qual_position` (63% of training rows, almost all before 2006) with the
  starting grid slot lowers the mean to 0.7215 (−0.0036, better in 1 of 3
  periods). `grid_fixed` is already a feature, so the fallback only repeats
  it and hides that those rows have no real qualifying result. We keep the
  median plus the `no_qual_time` flag.

**Decision.** Final features = `FEATURE_GROUPS` (12): `grid_fixed`;
`qual_position`, `gap_to_pole_pct`, `no_qual_time`, `qg_q3`, `qg_q2_only`;
`pts_before`, `pos_before`, `standings_missing`;
`cons_pts_before`, `cons_pos_before`, `cons_standings_missing`. Train on
1950–2019 with `sample_weight` (pre-2006 rows 0.25).